# NDScan Parameter Type Deep Dive

This notebook explores all the variations of parameter types and their specifications in detail.

In [1]:
import json
from collections import Counter, defaultdict

from sipyco import pyon

# Load and parse all experiments
explist = json.load(open("explist_debug.json", "r"))
experiments = explist["experiments"]

# Parse all ndscan experiments
all_schemata = {}  # fqn -> schema
for exp in experiments:
    if "ndscan_params" not in exp.get("arginfo", {}):
        continue
    ndscan_raw = exp["arginfo"]["ndscan_params"]
    if ndscan_raw[0] is None:
        continue
    parsed = pyon.decode(ndscan_raw[0]["default"])
    for fqn, schema in parsed.get("schemata", {}).items():
        all_schemata[fqn] = schema

print(f"Total unique parameters: {len(all_schemata)}")

Total unique parameters: 2695


## 1. Float Parameter Variations

In [2]:
# Collect all float schemas
float_schemas = {fqn: s for fqn, s in all_schemata.items() if s.get("type") == "float"}
print(f"Float parameters: {len(float_schemas)}")

# Analyze all spec keys
float_spec_keys = Counter()
for fqn, schema in float_schemas.items():
    for key in schema.get("spec", {}).keys():
        float_spec_keys[key] += 1

print("\nSpec key frequency:")
for key, count in float_spec_keys.most_common():
    pct = count / len(float_schemas) * 100
    print(f"  {key}: {count} ({pct:.1f}%)")

Float parameters: 2104

Spec key frequency:
  is_scannable: 2104 (100.0%)
  scale: 2104 (100.0%)
  step: 2104 (100.0%)
  unit: 1663 (79.0%)
  min: 906 (43.1%)
  max: 503 (23.9%)


In [3]:
# Group floats by unit
by_unit = defaultdict(list)
for fqn, schema in float_schemas.items():
    unit = schema.get("spec", {}).get("unit", "(unitless)")
    by_unit[unit].append((fqn, schema))

print(f"Unique units: {len(by_unit)}")
print("\nUnits and their frequency:")
for unit, params in sorted(by_unit.items(), key=lambda x: -len(x[1])):
    print(f"  {unit}: {len(params)} params")

Unique units: 17

Units and their frequency:
  us: 499 params
  (unitless): 441 params
  ms: 336 params
  A: 217 params
  kHz: 212 params
  MHz: 158 params
  V: 137 params
  s: 38 params
  Volts: 26 params
  mA: 19 params
  dB: 6 params
  Hz: 4 params
  mV: 4 params
  GHz: 3 params
  Ohms: 2 params
  THz: 1 params
  mHz: 1 params


In [4]:
# Example: MHz frequency parameters
print("=== MHz frequency parameters ===")
for fqn, schema in list(by_unit.get("MHz", []))[:3]:
    print(f"\n{schema['description']}:")
    print(f"  FQN: {fqn}")
    print(f"  Default: {schema['default']}")
    print(f"  Spec: {schema['spec']}")

=== MHz frequency parameters ===

Frequency for red_doublepass_injection:
  FQN: relock_ijd.RelockAllIJDsFrag.red_aom_frequency
  Default: 366900000.0
  Spec: {'is_scannable': True, 'scale': 1000000.0, 'step': 100000.0, 'min': 0, 'max': 500000000.0, 'unit': 'MHz'}

Frequency for blue_doublepass_injection:
  FQN: pyaion.fragments.default_beam_setter.SetBeamsToDefaults.frequency_blue_doublepass_injection
  Default: 200000000.0
  Spec: {'is_scannable': True, 'scale': 1000000.0, 'step': 100000.0, 'min': 0, 'max': 500000000.0, 'unit': 'MHz'}

Frequency for blue_singlepass_injection:
  FQN: pyaion.fragments.default_beam_setter.SetBeamsToDefaults.frequency_blue_singlepass_injection
  Default: 120000000.0
  Spec: {'is_scannable': True, 'scale': 1000000.0, 'step': 100000.0, 'min': 0, 'max': 500000000.0, 'unit': 'MHz'}


In [5]:
# Example: Time parameters (s, ms, us)
time_units = ["s", "ms", "us"]
for unit in time_units:
    if unit in by_unit:
        print(f"\n=== {unit} time parameters ===")
        for fqn, schema in list(by_unit[unit])[:2]:
            print(f"  {schema['description']}: default={schema['default']}, scale={schema['spec'].get('scale')}")


=== s time parameters ===
  v set wait time: default=0.3, scale=1.0
  v set wait time: default=0.3, scale=1.0

=== ms time parameters ===
  Time to wait after a current change: default=0, scale=0.001
  Time to wait between samples: default=0.001, scale=0.001

=== us time parameters ===
  wait time per scan step: default=0.002, scale=1e-06
  wait time per scan step: default=0.002, scale=1e-06


In [6]:
# Find floats with different constraint patterns
print("=== Float constraint patterns ===")

# With min and max
with_both = [(fqn, s) for fqn, s in float_schemas.items() if "min" in s.get("spec", {}) and "max" in s.get("spec", {})]
print(f"\nWith both min and max: {len(with_both)}")
if with_both:
    fqn, schema = with_both[0]
    print(f"  Example: {schema['description']}")
    print(f"  min={schema['spec']['min']}, max={schema['spec']['max']}")

# With only min
with_min_only = [
    (fqn, s) for fqn, s in float_schemas.items() if "min" in s.get("spec", {}) and "max" not in s.get("spec", {})
]
print(f"\nWith min only: {len(with_min_only)}")
if with_min_only:
    fqn, schema = with_min_only[0]
    print(f"  Example: {schema['description']}")
    print(f"  min={schema['spec']['min']}")

# With neither
no_bounds = [
    (fqn, s) for fqn, s in float_schemas.items() if "min" not in s.get("spec", {}) and "max" not in s.get("spec", {})
]
print(f"\nWith no bounds: {len(no_bounds)}")

=== Float constraint patterns ===

With both min and max: 406
  Example: v_min
  min=-4.0, max=4.0

With min only: 500
  Example: v set wait time
  min=0.0

With no bounds: 1101


## 2. Int Parameter Variations

In [7]:
# Collect all int schemas
int_schemas = {fqn: s for fqn, s in all_schemata.items() if s.get("type") == "int"}
print(f"Int parameters: {len(int_schemas)}")

# Analyze all spec keys
int_spec_keys = Counter()
for fqn, schema in int_schemas.items():
    for key in schema.get("spec", {}).keys():
        int_spec_keys[key] += 1

print("\nSpec key frequency:")
for key, count in int_spec_keys.most_common():
    pct = count / len(int_schemas) * 100
    print(f"  {key}: {count} ({pct:.1f}%)")

Int parameters: 261

Spec key frequency:
  is_scannable: 261 (100.0%)
  scale: 261 (100.0%)
  min: 261 (100.0%)
  max: 229 (87.7%)


In [8]:
# Show all int examples
print("=== All unique int parameters ===")
seen_descriptions = set()
for fqn, schema in int_schemas.items():
    desc = schema.get("description", "")
    if desc not in seen_descriptions:
        seen_descriptions.add(desc)
        print(f"\n{desc}:")
        print(f"  Default: {schema['default']}")
        print(f"  Spec: {schema['spec']}")
    if len(seen_descriptions) >= 10:
        break

=== All unique int parameters ===

n steps:
  Default: 100
  Spec: {'is_scannable': True, 'scale': 1, 'min': 10, 'max': 128}

Smoothing factor for averaging. Bigger = more smoothing:
  Default: 10000
  Spec: {'is_scannable': True, 'scale': 1, 'min': 0, 'max': 2147483648}

Number of voltage points to take, spaced by 1ms:
  Default: 10
  Spec: {'is_scannable': True, 'scale': 1, 'min': 1, 'max': 1000}

Number of scan points:
  Default: 100
  Spec: {'is_scannable': True, 'scale': 1, 'min': 0}

Sampler PGIA gain (0, 1, 2 or 3):
  Default: 0
  Spec: {'is_scannable': True, 'scale': 1, 'min': 0, 'max': 3}

Sampler channel to read:
  Default: 0
  Spec: {'is_scannable': True, 'scale': 1, 'min': 0, 'max': 7}

Max number of relock attempts:
  Default: 3
  Spec: {'is_scannable': True, 'scale': 1, 'min': 1}

PGA setting (0,1,2,3 == 1x,10x,100x,1000x):
  Default: 0
  Spec: {'is_scannable': True, 'scale': 1, 'min': 0, 'max': 3}

SUServo PGIA setting for blue_push_beam:
  Default: 0
  Spec: {'is_scanna

## 3. Bool Parameter Analysis

In [9]:
# Collect all bool schemas
bool_schemas = {fqn: s for fqn, s in all_schemata.items() if s.get("type") == "bool"}
print(f"Bool parameters: {len(bool_schemas)}")

# Analyze defaults
true_default = sum(1 for s in bool_schemas.values() if s.get("default") == "True")
false_default = sum(1 for s in bool_schemas.values() if s.get("default") == "False")
print(f"\nDefault True: {true_default} ({true_default/len(bool_schemas)*100:.1f}%)")
print(f"Default False: {false_default} ({false_default/len(bool_schemas)*100:.1f}%)")

# All bools should have is_scannable
scannable = sum(1 for s in bool_schemas.values() if s.get("spec", {}).get("is_scannable"))
print(f"\nScannable: {scannable}/{len(bool_schemas)}")

Bool parameters: 328

Default True: 156 (47.6%)
Default False: 172 (52.4%)

Scannable: 328/328


In [10]:
# Categorize bools by description pattern
enable_bools = []
other_bools = []

for fqn, schema in bool_schemas.items():
    desc = schema.get("description", "").lower()
    if "enable" in desc or "enabled" in desc:
        enable_bools.append((fqn, schema))
    else:
        other_bools.append((fqn, schema))

print(f"Enable/Enabled bools: {len(enable_bools)}")
print(f"Other bools: {len(other_bools)}")

print("\nSample 'other' bool descriptions:")
for fqn, schema in other_bools[:10]:
    print(f"  {schema['description']}")

Enable/Enabled bools: 70
Other bools: 258

Sample 'other' bool descriptions:
  Write settings
  Log if no relock
  Relock?
  Write settings
  Relock?
  Write settings
  Relock?
  Write settings
  Relock?
  Write settings


## 4. Schema Field Summary

In [11]:
# Top-level schema fields
schema_fields = Counter()
for fqn, schema in all_schemata.items():
    for key in schema.keys():
        schema_fields[key] += 1

print("Top-level schema fields:")
for field, count in schema_fields.most_common():
    pct = count / len(all_schemata) * 100
    print(f"  {field}: {count} ({pct:.1f}%)")

Top-level schema fields:
  fqn: 2695 (100.0%)
  description: 2695 (100.0%)
  type: 2695 (100.0%)
  default: 2695 (100.0%)
  spec: 2695 (100.0%)


In [12]:
# Canonical schema structure
print("""Canonical NDScan Parameter Schema Structure:

{
    "fqn": "<fully qualified name>",       # Always present
    "description": "<human readable>",     # Always present
    "type": "float" | "int" | "bool",      # Always present
    "default": "<string representation>",  # Always present (as string!)
    "spec": {                               # Always present
        "is_scannable": true,               # Always true for NDScan
        
        # For float/int:
        "scale": <number>,                  # Conversion factor (optional)
        "step": <number>,                   # UI increment (optional)
        "min": <number>,                    # Minimum value (optional)
        "max": <number>,                    # Maximum value (optional)
        "unit": "<string>"                  # Display unit (optional)
    }
}
""")

Canonical NDScan Parameter Schema Structure:

{
    "fqn": "<fully qualified name>",       # Always present
    "description": "<human readable>",     # Always present
    "type": "float" | "int" | "bool",      # Always present
    "default": "<string representation>",  # Always present (as string!)
    "spec": {                               # Always present
        "is_scannable": true,               # Always true for NDScan

        # For float/int:
        "scale": <number>,                  # Conversion factor (optional)
        "step": <number>,                   # UI increment (optional)
        "min": <number>,                    # Minimum value (optional)
        "max": <number>,                    # Maximum value (optional)
        "unit": "<string>"                  # Display unit (optional)
    }
}



## 5. FQN Naming Patterns

In [13]:
# Analyze FQN patterns
# FQN format: module.ClassName.parameter_name

fqn_parts = []
for fqn in all_schemata.keys():
    parts = fqn.split(".")
    fqn_parts.append(
        {
            "full": fqn,
            "num_parts": len(parts),
            "first": parts[0] if parts else "",
            "last": parts[-1] if parts else "",
        }
    )

# Number of parts distribution
num_parts_dist = Counter(fp["num_parts"] for fp in fqn_parts)
print("FQN parts distribution:")
for num, count in sorted(num_parts_dist.items()):
    print(f"  {num} parts: {count}")

FQN parts distribution:
  3 parts: 1897
  5 parts: 327
  6 parts: 130
  7 parts: 329
  9 parts: 4
  10 parts: 3
  11 parts: 5


In [14]:
# Show examples of each depth
for num_parts in sorted(num_parts_dist.keys()):
    examples = [fp for fp in fqn_parts if fp["num_parts"] == num_parts][:2]
    print(f"\n=== {num_parts} parts ===")
    for ex in examples:
        print(f"  {ex['full']}")


=== 3 parts ===
  relocker_board.AllRelockersFrag.blue_IJD1_relocker_enabled
  relocker_board.AllRelockersFrag.blue_IJD2_relocker_enabled

=== 5 parts ===
  repository.injected_diodes.set_koheron_controller.SetKoheronFrag.temperature
  repository.injected_diodes.set_koheron_controller.SetKoheronFrag.sampling_waittime

=== 6 parts ===
  relocker_board.AllRelockersFrag.build_fragment.<locals>._RelockerChannelFrag_blue_IJD1_relocker.v_min
  relocker_board.AllRelockersFrag.build_fragment.<locals>._RelockerChannelFrag_blue_IJD1_relocker.v_max

=== 7 parts ===
  repository.lib.fragments.beams.reset_all_beams.ResetAllICLBeams.enabled
  repository.lib.fragments.cameras.dual_camera_measurer.BGCorrectedMeasurement.save_raw_images

=== 9 parts ===
  repository.lib.fragments.read_adc.ReadSamplerADC_<artiq.master.worker_db.DummyDevice object at 0x7fffc6754070>_0.sampler_channel_gain
  repository.lib.fragments.read_adc.ReadSamplerADC_<artiq.master.worker_db.DummyDevice object at 0x7fffc673ee30>_1.s

## Summary Table

In [15]:
# Create summary table
print("=" * 70)
print("NDScan Parameter Type Summary")
print("=" * 70)
print(f"{'Type':<10} {'Count':<10} {'Common Spec Fields':<50}")
print("-" * 70)
print(f"{'float':<10} {len(float_schemas):<10} {'is_scannable, scale, step, unit, min, max':<50}")
print(f"{'int':<10} {len(int_schemas):<10} {'is_scannable, scale, min, max':<50}")
print(f"{'bool':<10} {len(bool_schemas):<10} {'is_scannable':<50}")
print("=" * 70)
print(f"{'TOTAL':<10} {len(all_schemata):<10}")

NDScan Parameter Type Summary
Type       Count      Common Spec Fields                                
----------------------------------------------------------------------
float      2104       is_scannable, scale, step, unit, min, max         
int        261        is_scannable, scale, min, max                     
bool       328        is_scannable                                      
TOTAL      2695      
